[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IOAI-official/IOAI-2026/blob/main/Individual-Contest/3_Potato/code/grading/evaluate.ipynb)

# Potato — evaluate a solution on Google Colab

Runs a solution the way the contest graded it, against the hidden leaderboard splits (now public in the [dataset](https://huggingface.co/datasets/IOAI-official/ioai-2026-potato)).

**Evaluating your own notebook:** upload it via the Files pane as `/content/solution.ipynb`, then Run all. Without an upload, the stock baseline is evaluated.

> **Unofficial educational version** — provided so the task can be used outside the contest environment, reading the data directly from the Hugging Face dataset. The official contest artifacts are preserved in `code/baseline-original/` and `code/grading-original/`.

In [ ]:
# ============================ Colab setup (added) ============================
# Downloads the task dataset from Hugging Face and lays it out exactly as the
# contest environment did. Everything below this cell is the original baseline.
import os, sys, shutil, subprocess
from pathlib import Path
def sh(c): print('+',c); subprocess.run(c, shell=True, check=True)
sh('pip -q install huggingface_hub')

from huggingface_hub import snapshot_download
DATA = Path(snapshot_download("IOAI-official/ioai-2026-potato", repo_type="dataset"))
def link(src,dst):
    dst=Path(dst); dst.parent.mkdir(parents=True,exist_ok=True)
    if dst.is_symlink() or dst.exists(): return
    os.symlink(src,dst)


In [ ]:
REPO = Path("/content/IOAI-2026")
if not REPO.exists(): sh(f"git clone --depth 1 https://github.com/IOAI-official/IOAI-2026 {REPO}")
BASE = REPO/"Individual-Contest/3_Potato/code/baseline-original"
ds = BASE/"dataset"; ds.mkdir(exist_ok=True)
for f in ("vocabulary.json","public_embeddings.npy","test_public.json"):
    if not (ds/f).exists(): os.symlink(DATA/"public"/f, ds/f)

SOL = Path("/content/solution.ipynb")
if not SOL.exists(): SOL = BASE/"solution.ipynb"
print("evaluating:", SOL)

# Full public suite with the task's own offline judge (approximate score):
sh(f"cd {BASE} && python local_test.py {SOL}")

# Contest-round replay: the hidden words of Leaderboard A, same public judge.
# (The official judge ranked with private embeddings; those are in the dataset's
#  private/ subset for study, but local_test.py judges with the public space.)
import json as _json
secrets = _json.loads((DATA/"private/secrets_leaderboard_a.json").read_text())
flags = " ".join(f"--secret {w}" for w in secrets)
sh(f"cd {BASE} && python local_test.py {SOL} {flags}")
